# QuantumCircuit.jl Phase 1 Static Core Walkthrough

**Audience**
- Julia users who want to try the current Phase 1 API

**Prerequisites**
- Basic Julia syntax
- Familiarity with superconducting-circuit terms such as transmon and resonator

**What you will learn**
- How to define a fixed transmon-resonator system
- How `CompositeSystem` groups subsystems and couplings
- How to build a QuantumToolbox-compatible Hamiltonian
- How to compute spectrum, transition frequencies, and anharmonicity


## Outline

1. Activate the local package environment
2. Build a minimal transmon-resonator `CompositeSystem`
3. Construct the model-layer Hamiltonian
4. Compute the low-lying spectrum
5. Interpret transition frequencies and anharmonicity
6. Try a small exercise by changing the coupling strength or truncation


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit

project_root


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


"/Users/yalgaeahn/Research/20_Projects/QuantumCircuit.jl/"

## Step 1 - Define a fixed transmon-resonator system

In Phase 1, the package supports fixed-parameter `Transmon`, `Resonator`, and `CapacitiveCoupling` objects.

**Unit convention**
- `EJ`, `EC`, resonator `ω`, and coupling `g` must use the same frequency unit.
- This notebook uses `GHz`, so the reported energies, transition frequencies, and anharmonicity are also in `GHz`.
- `ncut` and `dim` are dimensionless Hilbert-space truncation sizes.

The important design point is that the user assembles physics objects first and only later asks the Model layer to convert them into a Hamiltonian.


In [3]:
q1 = Transmon(:q1; EJ = 20.0, EC = 0.25, ncut = 6)
r1 = Resonator(:r1; ω = 6.8, dim = 5)
g1 = CapacitiveCoupling(:q1, :r1; g = 0.09)

sys = CompositeSystem(q1, r1, g1)

(
    subsystem_names = subsystem_names(sys),
    subsystem_count = length(subsystems(sys)),
    coupling_count = length(couplings(sys)),
)


(subsystem_names = [:q1, :r1], subsystem_count = 2, coupling_count = 1)

## Step 2 - Build the model-layer Hamiltonian

`build_model(sys)` keeps a small amount of structured metadata, and `hamiltonian(sys)` or `hamiltonian(model)` returns the QuantumToolbox-compatible object that the Simulation layer consumes.

For the current Phase 1 implementation, the transmon path uses a Duffing approximation derived from `EJ` and `EC`.

The current code assumes `\hbar = 1`, so the Hamiltonian entries and eigenvalues stay in the same frequency-unit convention as the input parameters.


## Step 2A - Hamiltonian used in the current code

The current implementation in `Model.jl` uses the following Hamiltonian pieces.

For a fixed transmon, the local Duffing approximation is

$$
\hat n = \hat a^\dagger \hat a
$$

$$
\omega_q = \sqrt{8 E_J E_C} - E_C, \qquad \alpha_q = -E_C
$$

$$
\hat H_q = \omega_q \hat n + \frac{\alpha_q}{2} \, \hat n (\hat n - I)
$$

For a resonator,

$$
\hat H_r = \omega_r \, \hat a^\dagger \hat a
$$

For the current `CapacitiveCoupling` implementation,

$$
\hat H_{\mathrm{int}} = g \left( \hat a_i^\dagger \hat a_j + \hat a_i \hat a_j^\dagger \right)
$$

So the total Hamiltonian assembled by the current code is

$$
\hat H = \sum_k \hat H_k + \sum_{(i,j)} \hat H_{\mathrm{int}}^{(i,j)}
$$

with each local operator embedded into the full Hilbert space by tensor products.

This is the implemented model, not yet a full circuit-quantization treatment.


In [4]:
model = build_model(sys)
H = hamiltonian(model)

(
    subsystem_order = model.subsystem_order,
    dimensions = model.dimensions,
    matrix_size = size(H.data),
    hamiltonian_type = typeof(H),
)


(subsystem_order = [:q1, :r1], dimensions = Dict(:r1 => 5, :q1 => 6), matrix_size = (30, 30), hamiltonian_type = QuantumToolbox.QuantumObject{QuantumToolbox.Operator, QuantumToolbox.ProductDimensions{2, 2, Tuple{QuantumToolbox.HilbertSpace, QuantumToolbox.HilbertSpace}, Tuple{QuantumToolbox.HilbertSpace, QuantumToolbox.HilbertSpace}}, SparseArrays.SparseMatrixCSC{ComplexF64, Int64}})

## Step 3 - Compute the low-lying spectrum

Now that the Hamiltonian is available, the Simulation layer can diagonalize it and return a typed `SpectrumResult`.


In [5]:
spec = spectrum(sys; levels = 6)
spec.energies


6-element Vector{Float64}:
  0.0
  6.063556513037353
  6.8109988072994
 11.882622095087275
 12.869092439153802
 13.621951426769206

## Step 4 - Derive transition frequencies and anharmonicity

The Analysis layer should work from result objects instead of talking directly to the solver. That is why `transition_frequencies` and `anharmonicity` take `SpectrumResult` as input.


In [6]:
(
    transition_frequencies = transition_frequencies(spec),
    anharmonicity = anharmonicity(spec),
)


(transition_frequencies = [6.063556513037353, 0.7474422942620471, 5.071623287787874, 0.986470344066527, 0.7528589876154044], anharmonicity = -5.316114218775306)

## Step 5 - Compare with a transmon-only system

A transmon-only example is useful because the Duffing approximation predicts an anharmonicity close to `-EC`.

For a coupled system, the spectrum is still valid, but the interpretation is less direct because the resonator hybridizes with the transmon.


In [7]:
transmon_only = CompositeSystem(Transmon(:q1; EJ = 20.0, EC = 0.25, ncut = 6))
transmon_spec = spectrum(transmon_only; levels = 4)

(
    energies = transmon_spec.energies,
    transition_frequencies = transition_frequencies(transmon_spec),
    anharmonicity = anharmonicity(transmon_spec),
)


(energies = [0.0, 6.074555320336759, 11.899110640673522, 17.473665961010273], transition_frequencies = [6.074555320336759, 5.824555320336763, 5.574555320336751], anharmonicity = -0.24999999999999645)

## Pitfalls and Extensions

**Common pitfall**
- Treating `ncut` or `dim` as cosmetic. These values set the Hilbert-space truncation and directly change the Hamiltonian size.

**Current limitation**
- `TunableTransmon` and `TunableCoupler` are not part of the current code path yet.

**Natural extension**
- After this notebook, the next useful addition is a flux-dependent example once the tunable subsystem types are implemented.


## Exercise

Modify the example below in one of two ways:

1. Increase the coupling strength `g` and compare the low-lying spectrum.
2. Increase the transmon truncation `ncut` and see how the energies change.

Before running the cell, predict whether the transition frequencies will move a little or a lot.


In [8]:
q1_ex = Transmon(:q1; EJ = 20.0, EC = 0.25, ncut = 8)
r1_ex = Resonator(:r1; ω = 6.8, dim = 5)
g1_ex = CapacitiveCoupling(:q1, :r1; g = 0.12)

sys_ex = CompositeSystem(q1_ex, r1_ex, g1_ex)
spec_ex = spectrum(sys_ex; levels = 6)

(
    energies = spec_ex.energies,
    transition_frequencies = transition_frequencies(spec_ex),
    anharmonicity = anharmonicity(spec_ex),
)


(energies = [0.0, 6.055220732717864, 6.819334587618892, 11.869959337596379, 12.86517407175932, 13.638532551654583], transition_frequencies = [6.055220732717864, 0.7641138549010273, 5.050624749977487, 0.9952147341629409, 0.7733584798952631], anharmonicity = -5.291106877816837)